In [ ]:
#setup
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate, stats

np.set_printoptions(precision=4, suppress=True)
pd.options.display.precision = 4
rng = np.random.default_rng(42)
print(sys.executable)

In [ ]:
#a discrete random variable, by formula and by simulation
values = np.arange(1, 7)
probs = np.full(6, 1 / 6)

mean = (values * probs).sum()
var = ((values - mean) ** 2 * probs).sum()
print(f"E[X] = {mean:.4f}, Var(X) = {var:.4f}, SD = {math.sqrt(var):.4f}")
print("shortcut E[X^2] - E[X]^2:", round((values ** 2 * probs).sum() - mean ** 2, 4))

rolls = rng.integers(1, 7, size=100_000)               # 1 to 6: the upper bound 7 is NOT included
print(f"simulated: mean {rolls.mean():.4f}, var {rolls.var():.4f}")

In [ ]:
#expected value of a game
net = np.array([490, -10])                             # a Rs 10 ticket: win Rs 500 (net 490), or lose the Rs 10
p = np.array([0.01, 0.99])
print("expected net gain per ticket:", (net * p).sum())      # -5.0: you lose Rs 5 per ticket on average

tickets = rng.choice(net, size=1_000_000, p=p)
print("simulated average:", tickets.mean().round(3))

In [ ]:
#rules for combining random variables
a = rng.integers(1, 7, size=500_000)
b = rng.integers(1, 7, size=500_000)
print("E[A+B]     :", (a + b).mean().round(3), "= 3.5 + 3.5")
print("Var(A+B)   :", (a + b).var().round(3), "= 2.9167 + 2.9167 = 5.8333   (A and B independent)")
print("Var(A+A)   :", (a + a).var().round(3), "= 4 x 2.9167 = 11.6667   (completely dependent)")
print("E[3A+2]    :", (3 * a + 2).mean().round(3), "= 3 x 3.5 + 2")
print("Var(3A+2)  :", (3 * a + 2).var().round(3), "= 9 x 2.9167 = 26.25   (the +2 does not change the spread)")

In [ ]:
#a discrete distribution to watch, the binomial
X = stats.binom(10, 0.5)                               # the number of heads in 10 fair flips
print("P(X = 5)  :", round(X.pmf(5), 4))               # 0.2461
print("P(X <= 3) :", round(X.cdf(3), 4))               # 0.1719
print("mean, var :", X.mean(), X.var())                # 5.0 and 2.5 (n*p and n*p*(1-p))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, (n, p) in zip(axes, ((5, 0.5), (20, 0.5), (200, 0.5))):
    ks = np.arange(0, n + 1)
    ax.bar(ks, stats.binom.pmf(ks, n, p), color="lightsteelblue")
    ax.set_title(f"binomial(n={n}, p={p})")
axes[2].set_xlim(70, 130)
plt.tight_layout()
plt.show()

In [ ]:
#density can be larger than 1
u = stats.uniform(loc=0, scale=0.5)                    # uniform on [0, 0.5]
print("pdf at 0.25            :", u.pdf(0.25))                          # 2.0: a density, not a probability
print("area under the curve   :", integrate.quad(u.pdf, 0, 0.5)[0])    # 1.0
print("P(X = 0.25)            : exactly 0, but P(0.24 < X < 0.26) =", round(u.cdf(0.26) - u.cdf(0.24), 4))

In [ ]:
#the PDF, CDF, and area side by side
x = np.linspace(-4, 4, 500)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(x, stats.norm.pdf(x))
inside = x <= 1
axes[0].fill_between(x[inside], stats.norm.pdf(x[inside]), color="lightsteelblue")
axes[0].set_title(f"PDF: shaded area = P(Z <= 1) = {stats.norm.cdf(1):.4f}")
axes[1].plot(x, stats.norm.cdf(x))
axes[1].axvline(1, color="gray", ls="--")
axes[1].axhline(stats.norm.cdf(1), color="gray", ls="--")
axes[1].set_title("CDF: the height at 1 is that same area")
plt.tight_layout()
plt.show()

In [ ]:
#waiting for a bus that comes every 15 minutes
wait = stats.uniform(loc=0, scale=15)                  # you arrive at a random moment
print("mean wait       :", wait.mean())                # 7.5
print("SD              :", round(wait.std(), 3))       # 4.33 = 15 / sqrt(12)
print("P(wait > 10)    :", round(wait.sf(10), 4))      # 0.3333
print("P(3 < wait < 6) :", round(wait.cdf(6) - wait.cdf(3), 4))     # 0.2
print("25th percentile :", wait.ppf(0.25))            # 3.75

In [ ]:
#the loc and scale trap, and checking the formulas
print(stats.uniform(5, 10).support())                  # (5.0, 15.0): loc=5, scale=10 means [5, 15], NOT [5, 10]
print(stats.uniform(5, 5).support())                   # (5.0, 10.0)
print(rng.uniform(5, 10, 3))                           # numpy's uniform takes the LOW and HIGH bounds directly

low, high = 2, 8
samples = rng.uniform(low, high, 100_000)
print(f"mean {samples.mean():.3f} (formula {(low + high) / 2}), var {samples.var():.3f} (formula {(high - low) ** 2 / 12})")

In [ ]:
#IQ scores, N(100, 15)
iq = stats.norm(loc=100, scale=15)
print("P(IQ > 130)      :", round(iq.sf(130), 5))                  # 0.02275
print("P(IQ < 70)       :", round(iq.cdf(70), 5))                  # 0.02275 (symmetric)
print("P(85 < IQ < 115) :", round(iq.cdf(115) - iq.cdf(85), 4))    # 0.6827
print("90th percentile  :", round(iq.ppf(0.90), 2))                # 119.22
print("top 5% start at  :", round(iq.ppf(0.95), 2))                # 124.67
print("middle 95%       :", np.round(iq.interval(0.95), 2))        # [ 70.6 129.4]
print("pdf at the mean  :", round(iq.pdf(100), 4))                 # 0.0266 = 1 / (15 x sqrt(2 pi))

z = (130 - 100) / 15
print(f"standardised: z = {z}, the standard normal tail gives {stats.norm.sf(z):.5f}   (the same answer)")
for zz in (-3, -2, -1, 0, 1, 1.645, 1.96, 2, 3):
    print(f"  P(Z <= {zz:>6}) = {stats.norm.cdf(zz):.4f}")

In [ ]:
#the effect of μ and σ, and a shaded probability
x = np.linspace(-10, 20, 600)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for mu, sd in ((0, 1), (0, 2), (5, 1)):
    axes[0].plot(x, stats.norm.pdf(x, mu, sd), label=f"mean {mu}, sd {sd}")
axes[0].legend()
axes[0].set_title("The mean moves the curve, the SD stretches it")

xs = np.linspace(55, 145, 500)
axes[1].plot(xs, iq.pdf(xs))
band = (xs >= 85) & (xs <= 115)
axes[1].fill_between(xs[band], iq.pdf(xs[band]), color="lightsteelblue")
axes[1].set_title("P(85 < IQ < 115) = shaded area = 0.6827")
plt.tight_layout()
plt.show()

In [ ]:
#is my data normal? Histogram, Q-Q plot, and fitting
normal_data = rng.normal(65, 10, 1000)
skewed_data = rng.exponential(10, 1000)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for row, (name, data) in enumerate((("normal data", normal_data), ("exponential data", skewed_data))):
    axes[row, 0].hist(data, bins=40, density=True, color="lightsteelblue", edgecolor="white")
    axes[row, 0].set_title(name)
    stats.probplot(data, dist="norm", plot=axes[row, 1])
    axes[row, 1].set_title(f"{name}: Q-Q plot")
plt.tight_layout()
plt.show()

mu_hat, sd_hat = stats.norm.fit(normal_data)           # estimate the parameters from the data
print(f"fitted mean {mu_hat:.2f}, fitted sd {sd_hat:.2f}   (true values 65 and 10)")
print("skew:", round(stats.skew(normal_data), 2), "vs", round(stats.skew(skewed_data), 2))

In [ ]:
#customers arriving every 5 minutes on average
mean_gap = 5
lam = 1 / mean_gap                                     # rate: 0.2 customers per minute
gap = stats.expon(scale=mean_gap)                      # scipy's scale is the MEAN (1 / rate), not the rate

print("mean, sd, median :", gap.mean(), gap.std(), round(gap.median(), 4))     # 5.0 5.0 3.4657
print("P(wait > 10 min) :", round(gap.sf(10), 4), "=", round(math.exp(-lam * 10), 4))   # 0.1353
print("P(wait < 3 min)  :", round(gap.cdf(3), 4))                                # 0.4512
print("90th percentile  :", round(gap.ppf(0.9), 3))                              # 11.513